In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

DATA_DIR = '../dataviz'
PRODUCT = 'INTARIAN_PEPPER_ROOT'

In [ ]:

# Load all price CSVs
price_files = sorted(glob.glob(f'{DATA_DIR}/prices_round_1_day_*.csv'))
prices = pd.concat(
    [pd.read_csv(f, sep=';') for f in price_files],
    ignore_index=True
)
prices = prices[prices['product'] == PRODUCT].copy()
prices['global_ts'] = prices['day'] * 1_000_000 + prices['timestamp']
prices = prices.sort_values('global_ts').reset_index(drop=True)

# Load all trade CSVs
trade_files = sorted(glob.glob(f'{DATA_DIR}/trades_round_1_day_*.csv'))
trades_raw = []
for f in trade_files:
    day_val = int(os.path.basename(f).split('day_')[1].replace('.csv', ''))
    df = pd.read_csv(f, sep=';')
    df['day'] = day_val
    trades_raw.append(df)
trades = pd.concat(trades_raw, ignore_index=True)
trades = trades[trades['symbol'] == PRODUCT].copy()
trades['global_ts'] = trades['day'] * 1_000_000 + trades['timestamp']
trades = trades.sort_values('global_ts').reset_index(drop=True)

# Filter
PRICE_FLOOR = 8000
prices = prices[prices['mid_price'] >= PRICE_FLOOR].reset_index(drop=True)
trades = trades[trades['price'] >= PRICE_FLOOR].reset_index(drop=True)

print('prices:', prices.shape, '  trades:', trades.shape)

# Plot all filtered data
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for day in sorted(prices['day'].unique()):
    d = prices[prices['day'] == day]
    axes[0].plot(d['global_ts'], d['mid_price'], label=f'day {day}')
axes[0].set_title(f'{PRODUCT} — Mid Price (>= {PRICE_FLOOR})')
axes[0].legend()

axes[1].scatter(trades['global_ts'], trades['price'], s=8, color='red', label='trades')
axes[1].set_title('Trade Prices')
axes[1].legend()

spread = prices['ask_price_1'] - prices['bid_price_1']
axes[2].plot(prices['global_ts'], spread, color='purple')
axes[2].set_title('Bid-Ask Spread')

plt.tight_layout()
plt.show()


In [ ]:
# Fit linear slope to mid_price over global_ts
x = prices['global_ts'].values
y = prices['mid_price'].values

slope, intercept = np.polyfit(x, y, 1)
print(f'slope:     {slope:.6e}  (price units per timestamp unit)')
print(f'intercept: {intercept:.4f}')

# Detrend prices
trend = slope * x + intercept
prices['detrended_mid']   = prices['mid_price']   - trend
prices['detrended_bid1']  = prices['bid_price_1']  - trend
prices['detrended_ask1']  = prices['ask_price_1']  - trend

# Detrend trades (interpolate trend at trade timestamps)
trade_trend = slope * trades['global_ts'].values + intercept
trades['detrended_price'] = trades['price'] - trade_trend

# Visual: raw mid + fitted line
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(x, y, alpha=0.6, label='mid price')
ax.plot(x, trend, color='red', linewidth=2, label=f'fit  slope={slope:.4e}')
ax.set_title(f'{PRODUCT} — Raw Mid Price + Linear Fit')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Detrended mid price centred on 0
fig, ax = plt.subplots(figsize=(14, 5))

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
for day in sorted(prices['day'].unique()):
    d = prices[prices['day'] == day]
    ax.plot(d['global_ts'], d['detrended_mid'], label=f'day {day}')

ax.set_title(f'{PRODUCT} — Detrended Mid Price (mean≈0)')
ax.set_ylabel('price − trend')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Detrended mid price + detrended trade prices
fig, ax = plt.subplots(figsize=(14, 5))

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.plot(prices['global_ts'], prices['detrended_mid'], alpha=0.7, label='mid (detrended)')
ax.scatter(trades['global_ts'], trades['detrended_price'],
           color='red', s=12, zorder=5, label='trades (detrended)')

ax.set_title(f'{PRODUCT} — Detrended Mid + Trades')
ax.set_ylabel('price − trend')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Detrended bid/ask + spread
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].plot(prices['global_ts'], prices['detrended_bid1'], label='bid1', alpha=0.7)
axes[0].plot(prices['global_ts'], prices['detrended_mid'],  label='mid',  alpha=0.9)
axes[0].plot(prices['global_ts'], prices['detrended_ask1'], label='ask1', alpha=0.7)
axes[0].set_title('Detrended Bid / Mid / Ask')
axes[0].legend()

spread = prices['ask_price_1'] - prices['bid_price_1']
axes[1].plot(prices['global_ts'], spread, color='purple')
axes[1].set_title('Bid-Ask Spread (raw, not detrended)')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of detrended mid price
det = prices['detrended_mid'].dropna()
print(f'mean:  {det.mean():.4f}')
print(f'std:   {det.std():.4f}')
print(f'min:   {det.min():.4f}  max: {det.max():.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(det, bins=60, edgecolor='black')
ax.axvline(0, color='red', linewidth=1.5, linestyle='--', label='0')
ax.set_title('Detrended Mid Price Distribution')
ax.legend()
plt.tight_layout()
plt.show()